# CVE Data - Exploratory Data Analysis

**Purpose**: Comprehensive exploration and visualization of CVE data with enrichments

**What this notebook does**:
1. **Data Loading** - Load CVEs from database with enrichments
2. **Quality Checks** - Verify data completeness and integrity
3. **Temporal Analysis** - CVE publication trends over time
4. **Severity Analysis** - CVSS score distributions
5. **Risk Signals** - KEV, EPSS, Healthcare, ATT&CK, CHPL coverage
6. **Feature Correlations** - Relationships between risk indicators
7. **Label Distribution** - Weak label analysis

**Note**: All plots are saved externally to `outputs/plots/` to keep notebook size minimal.

---

## 1. Setup & Imports

In [45]:
import sys
import os
from pathlib import Path
from datetime import datetime, timedelta
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# Setup project paths
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(project_root))

# Import project modules
from src.core.cve_database import CVEDatabase
from src.utils.notebook_helpers import save_plot, display_sample, setup_notebook_output
from config.settings import settings

# Configure notebook display
setup_notebook_output()
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print(f"[OK] Project root: {project_root}")
print(f"[OK] Database: {settings.get_database_path()}")
print(f"[OK] Imports successful")


[OK] Notebook output configured
[OK] Project root: /Users/vinayksharma/AirDnd/cti_recommender
[OK] Database: /Users/vinayksharma/AirDnd/cti_recommender/data/cve_database.db
[OK] Imports successful


## 2. Data Loading

In [46]:
# Connect to database
db = CVEDatabase()

# Get database statistics
stats = db.get_statistics()

# Extract date range from tuple
earliest_date, latest_date = stats.get('date_range', (None, None))

print("="*70)
print("DATABASE OVERVIEW")
print("="*70)
print(f"Total CVEs: {stats['total_cves']:,}")
print(f"Date range: {earliest_date} to {latest_date}")
print("="*70)

2026-03-08 11:02:03 - src.core.cve_database - INFO - Connected to database
2026-03-08 11:02:03 - src.core.cve_database - INFO - Database schema created/verified
DATABASE OVERVIEW
Total CVEs: 226,320
Date range: 2018-01-01T00:29:00.213 to 2025-12-31T23:15:42.413


In [47]:
# Load CVE data with ALL enrichments - FULL DATASET for thesis
# This includes both external signals (16) and computed features (37) = 53 total enrichment features
query = """
SELECT 
    c.cve_id,
    c.published,
    c.modified,
    c.description,
    c.cvss,
    c.cvss_vector,
    c.cwe,
    -- External enrichment signals (15 features)
    e.kev_flag,
    e.epss_score,
    e.epss_percentile,
    e.is_healthcare,
    e.healthcare_score,
    e.attack_flag,
    e.attack_technique_count,
    e.chpl_flag,
    e.is_curated,
    e.label,
    -- CVSS Decomposition features (10)
    e.cvss_av,
    e.cvss_ac,
    e.cvss_pr,
    e.cvss_ui,
    e.cvss_s,
    e.cvss_c,
    e.cvss_i,
    e.cvss_a,
    e.cvss_score_derived,
    e.cvss_severity_category,
    -- CWE Intelligence features (8)
    e.cwe_is_top25,
    e.cwe_is_injection,
    e.cwe_is_crypto,
    e.cwe_is_access_control,
    e.cwe_is_input_validation,
    e.cwe_is_memory_corruption,
    e.cwe_category,
    e.cwe_severity_score,
    -- Description NLP features (10)
    e.desc_has_rce,
    e.desc_has_auth_bypass,
    e.desc_has_priv_esc,
    e.desc_has_sqli,
    e.desc_has_xss,
    e.desc_has_dos,
    e.desc_has_buffer_overflow,
    e.desc_has_path_traversal,
    e.desc_has_csrf,
    e.desc_has_xxe,
    -- Vendor features (3)
    e.vendor_is_high_risk,
    e.vendor_is_healthcare,
    e.vendor_risk_score,
    -- Interaction features (6)
    e.ultimate_risk,
    e.critical_exploitable,
    e.network_accessible,
    e.auth_not_required,
    e.high_impact_network,
    e.healthcare_critical
FROM cves c
LEFT JOIN enrichments e ON c.cve_id = e.cve_id
WHERE c.cvss IS NOT NULL
ORDER BY c.published DESC
"""

df = pd.read_sql(query, db.conn)
df['published'] = pd.to_datetime(df['published'])
df['modified'] = pd.to_datetime(df['modified'])

print(f"\n[OK] Loaded {len(df):,} CVEs (FULL DATASET)")
print(f"  Date range: {df['published'].min().date()} to {df['published'].max().date()}")
print(f"  Time span: {(df['published'].max() - df['published'].min()).days} days")

# Display sample
display_sample(df, n=10, title="Sample CVE Records")


[OK] Loaded 210,147 CVEs (FULL DATASET)
  Date range: 2018-01-01 to 2025-12-31
  Time span: 2921 days


Showing 10 of 210,147 rows


,cve_id,published,modified,description,cvss,cvss_vector,cwe,...,vendor_risk_score,ultimate_risk,critical_exploitable,network_accessible,auth_not_required,high_impact_network,healthcare_critical
0,CVE-2025-67711,2025-12-31 23:15:42.413,2026-01-06 19:03:34.700,There is a stored cross site scripting issue i...,6.1,CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:C/C:L/I:L/A:N,CWE-79,...,0.571429,1.666667,0,1,1,0,0
1,CVE-2025-67710,2025-12-31 23:15:42.270,2026-01-06 19:04:06.150,There is a stored cross site scripting issue i...,6.1,CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:C/C:L/I:L/A:N,CWE-79,...,0.571429,1.666667,0,1,1,0,0
2,CVE-2025-67709,2025-12-31 23:15:42.130,2026-01-06 19:04:27.810,There is a stored cross site scripting issue i...,6.1,CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:C/C:L/I:L/A:N,CWE-79,...,0.571429,1.666667,0,1,1,0,0
3,CVE-2025-67708,2025-12-31 23:15:41.980,2026-01-06 19:04:52.547,There is a stored cross site scripting issue i...,6.1,CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:C/C:L/I:L/A:N,CWE-79,...,0.571429,1.666667,0,1,1,0,0
4,CVE-2025-67707,2025-12-31 23:15:41.833,2026-01-06 19:08:02.547,ArcGIS Server version 11.5 and earlier on Wind...,5.6,CVSS:3.1/AV:N/AC:H/PR:N/UI:N/S:U/C:L/I:L/A:L,CWE-434,...,0.571429,2.333333,0,1,1,1,0
5,CVE-2025-67706,2025-12-31 23:15:41.687,2026-01-06 19:08:47.110,ArcGIS Server version 11.5 and earlier on Wind...,5.6,CVSS:3.1/AV:N/AC:H/PR:N/UI:N/S:U/C:L/I:L/A:L,CWE-434,...,0.571429,2.333333,0,1,1,1,0
6,CVE-2025-67705,2025-12-31 23:15:41.540,2026-01-06 19:09:08.807,There is a stored cross site scripting issue i...,6.1,CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:C/C:L/I:L/A:N,CWE-79,...,0.571429,1.666667,0,1,1,0,0
7,CVE-2025-67704,2025-12-31 23:15:41.387,2026-01-06 19:14:39.267,There is a stored cross site scripting issue i...,6.1,CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:C/C:L/I:L/A:N,CWE-79,...,0.571429,1.666667,0,1,1,0,0
8,CVE-2025-67703,2025-12-31 23:15:40.540,2026-01-06 19:15:11.537,There is a stored cross site scripting issue i...,6.1,CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:C/C:L/I:L/A:N,CWE-79,...,0.571429,1.666667,0,1,1,0,0
9,CVE-2025-69288,2025-12-31 22:15:49.410,2026-01-13 15:25:44.200,Titra is open source project time tracking sof...,9.1,CVSS:3.1/AV:N/AC:L/PR:H/UI:N/S:C/C:H/I:H/A:H,CWE-20,...,0.000000,0.000000,1,1,0,1,0


... 210,137 more rows


## 3. Data Quality Checks

In [48]:
print("="*70)
print("DATA QUALITY ASSESSMENT")
print("="*70)

# Completeness check
print("\n[STATS] Completeness:")
completeness = {
    'CVE ID': (df['cve_id'].notna().sum() / len(df)) * 100,
    'Published': (df['published'].notna().sum() / len(df)) * 100,
    'CVSS': (df['cvss'].notna().sum() / len(df)) * 100,
    'CWE': (df['cwe'].notna().sum() / len(df)) * 100,
    'EPSS': (df['epss_score'].notna().sum() / len(df)) * 100,
    'Description': (df['description'].notna().sum() / len(df)) * 100
}

for field, pct in completeness.items():
    status = "[OK]" if pct >= 90 else "[WARN]" if pct >= 70 else "[FAIL]"
    print(f"  {status} {field:15s}: {pct:5.1f}%")

# Enrichment signal coverage
print("\n[TARGET] Enrichment Signals:")
signals = {
    'KEV (exploited)': df['kev_flag'].sum(),
    'Healthcare-related': df['is_healthcare'].sum(),
    'ATT&CK mapped': df['attack_flag'].sum(),
    'CHPL certified': df['chpl_flag'].sum(),
    'Curated breaches': df['is_curated'].sum()
}

for signal, count in signals.items():
    pct = (count / len(df)) * 100
    print(f"  {signal:25s}: {count:6,} ({pct:5.2f}%)")

# Label distribution
if 'label' in df.columns and df['label'].notna().sum() > 0:
    print("\n  Label Distribution:")
    label_dist = df['label'].value_counts().sort_index()
    for label, count in label_dist.items():
        pct = (count / len(df)) * 100
        print(f"  Label {label}: {count:6,} ({pct:5.2f}%)")

print("\n" + "="*70)

DATA QUALITY ASSESSMENT

[STATS] Completeness:
  [OK] CVE ID         : 100.0%
  [OK] Published      : 100.0%
  [OK] CVSS           : 100.0%
  [OK] CWE            :  99.0%
  [OK] EPSS           : 100.0%
  [OK] Description    : 100.0%

[TARGET] Enrichment Signals:
  KEV (exploited)          :  1,177 ( 0.56%)
  Healthcare-related       :    796 ( 0.38%)
  ATT&CK mapped            : 81,534 (38.80%)
  CHPL certified           :  4,951 ( 2.36%)
  Curated breaches         :     52 ( 0.02%)

  Label Distribution:
  Label 0: 10,577 ( 5.03%)
  Label 1: 34,547 (16.44%)
  Label 2: 152,122 (72.39%)
  Label 3: 12,635 ( 6.01%)
  Label 4:    266 ( 0.13%)



## 4. Temporal Analysis

In [49]:
# CVE publication trends over time
df_monthly = df.groupby(df['published'].dt.to_period('M')).size().reset_index()
df_monthly.columns = ['month', 'count']
df_monthly['month'] = df_monthly['month'].dt.to_timestamp()

fig = px.line(
    df_monthly,
    x='month',
    y='count',
    title='CVE Publications Over Time (Monthly)',
    labels={'month': 'Month', 'count': 'Number of CVEs'}
)
fig.update_traces(line_color='#2E86C1', line_width=2)
fig.update_layout(height=400, showlegend=False)

# Save externally
save_plot(fig, 'temporal_trends_monthly')

# Show summary statistics
print(f"\n Temporal Statistics:")
print(f"  Total months: {len(df_monthly)}")
print(f"  Mean CVEs/month: {df_monthly['count'].mean():.1f}")
print(f"  Peak month: {df_monthly.loc[df_monthly['count'].idxmax(), 'month'].strftime('%Y-%m')} ({df_monthly['count'].max():,} CVEs)")
print(f"  Recent 3 months avg: {df_monthly.tail(3)['count'].mean():.1f} CVEs/month")


 Temporal Statistics:
  Total months: 96
  Mean CVEs/month: 2189.0
  Peak month: 2024-05 (4,971 CVEs)
  Recent 3 months avg: 3477.0 CVEs/month


In [50]:
# CVE publications by year
df_yearly = df.groupby(df['published'].dt.year).agg({
    'cve_id': 'count',
    'kev_flag': 'sum',
    'is_healthcare': 'sum',
    'attack_flag': 'sum'
}).reset_index()
df_yearly.columns = ['year', 'total_cves', 'kev_cves', 'healthcare_cves', 'attack_cves']

fig = go.Figure()
fig.add_trace(go.Bar(x=df_yearly['year'], y=df_yearly['total_cves'], name='Total CVEs'))
fig.add_trace(go.Bar(x=df_yearly['year'], y=df_yearly['kev_cves'], name='KEV'))
fig.add_trace(go.Bar(x=df_yearly['year'], y=df_yearly['healthcare_cves'], name='Healthcare'))

fig.update_layout(
    title='CVE Distribution by Year with Risk Signals',
    xaxis_title='Year',
    yaxis_title='Number of CVEs',
    barmode='group',
    height=450
)

save_plot(fig, 'cve_by_year_with_signals')

print("\nYearly Summary:")
display_sample(df_yearly, n=20, title="CVEs by Year")


Yearly Summary:


Showing 20 of 8 rows


,year,total_cves,kev_cves,healthcare_cves,attack_cves
0,2018,16510,76,77,5650
1,2019,17305,128,81,5553
2,2020,18322,145,76,6319
3,2021,20149,210,74,7605
4,2022,25062,129,148,9816
5,2023,28817,160,45,11528
6,2024,39618,155,134,16942
7,2025,44364,174,161,18121


### 4.1 CVSS Score Temporal Trends

CVE volume growth alongside average CVSS score evolution over time.

In [51]:
# CVE Distribution by Year with CVSS Score Trends
# Query year-by-year CVE distribution with CVSS statistics
conn = db.conn
query = """
SELECT 
    strftime('%Y', published) as year,
    COUNT(*) as total_cves,
    COUNT(cvss) as has_cvss,
    ROUND(AVG(cvss), 2) as avg_cvss
FROM cves
WHERE published IS NOT NULL
GROUP BY year
ORDER BY year
"""

df_cvss_temporal = pd.read_sql_query(query, conn)

# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add CVE count bars
fig.add_trace(
    go.Bar(
        x=df_cvss_temporal['year'],
        y=df_cvss_temporal['total_cves'],
        name='Total CVEs',
        marker_color='lightblue',
        text=df_cvss_temporal['total_cves'],
        texttemplate='%{text:,}',
        textposition='outside'
    ),
    secondary_y=False
)

# Add average CVSS line
fig.add_trace(
    go.Scatter(
        x=df_cvss_temporal['year'],
        y=df_cvss_temporal['avg_cvss'],
        name='Average CVSS Score',
        mode='lines+markers',
        line=dict(color='red', width=3),
        marker=dict(size=10, symbol='diamond'),
        text=df_cvss_temporal['avg_cvss'],
        texttemplate='%{text}',
        textposition='top center'
    ),
    secondary_y=True
)

# Update axes
fig.update_xaxes(title_text="Publication Year", type='category')
fig.update_yaxes(title_text="<b>Number of CVEs</b>", secondary_y=False)
fig.update_yaxes(
    title_text="<b>Average CVSS Score</b>", 
    secondary_y=True,
    range=[0, 10]
)

# Update layout
fig.update_layout(
    title='CVE Volume Growth and CVSS Score Trends (2018-2025)',
    height=500,
    hovermode='x unified',
    showlegend=True
)

save_plot(fig, 'cvss_temporal_trends')

# Print summary statistics
print("\n[STATS] CVSS Temporal Trends:")
print(f"  Total CVEs: {df_cvss_temporal['total_cves'].sum():,}")
print(f"  Years covered: {df_cvss_temporal['year'].min()} - {df_cvss_temporal['year'].max()}")
print(f"  CVE growth: {df_cvss_temporal['total_cves'].iloc[0]:,} (2018) → {df_cvss_temporal['total_cves'].iloc[-1]:,} (2025)")
print(f"  Growth rate: {((df_cvss_temporal['total_cves'].iloc[-1] / df_cvss_temporal['total_cves'].iloc[0]) - 1) * 100:.1f}%")
print(f"\n  CVSS Score Trend:")
print(f"  2018 avg: {df_cvss_temporal['avg_cvss'].iloc[0]:.2f}")
print(f"  2025 avg: {df_cvss_temporal['avg_cvss'].iloc[-1]:.2f}")
print(f"  Change: {df_cvss_temporal['avg_cvss'].iloc[-1] - df_cvss_temporal['avg_cvss'].iloc[0]:.2f} ({((df_cvss_temporal['avg_cvss'].iloc[-1] / df_cvss_temporal['avg_cvss'].iloc[0]) - 1) * 100:.1f}%)")



[STATS] CVSS Temporal Trends:
  Total CVEs: 226,320
  Years covered: 2018 - 2025
  CVE growth: 18,154 (2018) → 49,972 (2025)
  Growth rate: 175.3%

  CVSS Score Trend:
  2018 avg: 7.30
  2025 avg: 6.60
  Change: -0.70 (-9.6%)


## 5. CVSS Score Analysis

In [52]:
# CVSS distribution
fig = px.histogram(
    df,
    x='cvss',
    nbins=50,
    title='CVSS Score Distribution',
    labels={'cvss': 'CVSS Score', 'count': 'Number of CVEs'},
    color_discrete_sequence=['#E74C3C']
)
fig.add_vline(x=7.0, line_dash="dash", line_color="orange", annotation_text="High (7.0)")
fig.add_vline(x=9.0, line_dash="dash", line_color="red", annotation_text="Critical (9.0)")
fig.update_layout(height=400)

save_plot(fig, 'cvss_distribution')

# CVSS statistics
print("\n[STATS] CVSS Statistics:")
print(f"  Mean: {df['cvss'].mean():.2f}")
print(f"  Median: {df['cvss'].median():.2f}")
print(f"  Std Dev: {df['cvss'].std():.2f}")
print(f"\n  Severity Distribution:")
print(f"    Low (0.0-3.9):    {((df['cvss'] < 4.0).sum()):6,} ({(df['cvss'] < 4.0).mean()*100:5.1f}%)")
print(f"    Medium (4.0-6.9): {((df['cvss'] >= 4.0) & (df['cvss'] < 7.0)).sum():6,} ({((df['cvss'] >= 4.0) & (df['cvss'] < 7.0)).mean()*100:5.1f}%)")
print(f"    High (7.0-8.9):   {((df['cvss'] >= 7.0) & (df['cvss'] < 9.0)).sum():6,} ({((df['cvss'] >= 7.0) & (df['cvss'] < 9.0)).mean()*100:5.1f}%)")
print(f"    Critical (9.0+):  {(df['cvss'] >= 9.0).sum():6,} ({(df['cvss'] >= 9.0).mean()*100:5.1f}%)")


[STATS] CVSS Statistics:
  Mean: 6.88
  Median: 6.80
  Std Dev: 1.72

  Severity Distribution:
    Low (0.0-3.9):     7,728 (  3.7%)
    Medium (4.0-6.9): 97,514 ( 46.4%)
    High (7.0-8.9):   79,674 ( 37.9%)
    Critical (9.0+):  25,231 ( 12.0%)


## 6. EPSS Score Analysis

In [53]:
# EPSS distribution (log scale)
df_epss = df[df['epss_score'].notna()].copy()

fig = px.histogram(
    df_epss,
    x='epss_score',
    nbins=100,
    title='EPSS Score Distribution (Exploit Probability)',
    labels={'epss_score': 'EPSS Score', 'count': 'Number of CVEs'},
    log_y=True,
    color_discrete_sequence=['#16A085']
)
fig.add_vline(x=0.1, line_dash="dash", line_color="orange", annotation_text="10% threshold")
fig.add_vline(x=0.5, line_dash="dash", line_color="red", annotation_text="50% threshold")
fig.update_layout(height=400)

save_plot(fig, 'epss_distribution')

# EPSS statistics
print("\n[STATS] EPSS Statistics:")
print(f"  CVEs with EPSS: {len(df_epss):,} ({len(df_epss)/len(df)*100:.1f}%)")
print(f"  Mean: {df_epss['epss_score'].mean():.4f}")
print(f"  Median: {df_epss['epss_score'].median():.4f}")
print(f"  95th percentile: {df_epss['epss_score'].quantile(0.95):.4f}")
print(f"\n  Risk Categories:")
print(f"    EPSS < 0.1:   {(df_epss['epss_score'] < 0.1).sum():6,} ({(df_epss['epss_score'] < 0.1).mean()*100:5.1f}%)")
print(f"    EPSS 0.1-0.5: {((df_epss['epss_score'] >= 0.1) & (df_epss['epss_score'] < 0.5)).sum():6,} ({((df_epss['epss_score'] >= 0.1) & (df_epss['epss_score'] < 0.5)).mean()*100:5.1f}%)")
print(f"    EPSS >= 0.5:  {(df_epss['epss_score'] >= 0.5).sum():6,} ({(df_epss['epss_score'] >= 0.5).mean()*100:5.1f}%)")


[STATS] EPSS Statistics:
  CVEs with EPSS: 210,147 (100.0%)
  Mean: 0.0254
  Median: 0.0019
  95th percentile: 0.0886

  Risk Categories:
    EPSS < 0.1:   200,316 ( 95.3%)
    EPSS 0.1-0.5:  6,267 (  3.0%)
    EPSS >= 0.5:   3,564 (  1.7%)


## 7. CVSS vs EPSS Relationship

In [54]:
# Scatter plot: CVSS vs EPSS
df_plot = df[df['epss_score'].notna()].sample(min(5000, len(df_epss)))  # Sample for performance

fig = px.scatter(
    df_plot,
    x='cvss',
    y='epss_score',
    color='kev_flag',
    title='CVSS vs EPSS Score (Sample)',
    labels={'cvss': 'CVSS Score', 'epss_score': 'EPSS Score', 'kev_flag': 'KEV Listed'},
    opacity=0.6,
    color_continuous_scale='RdYlGn_r'
)
fig.update_layout(height=500)

save_plot(fig, 'cvss_vs_epss_scatter')

# Correlation
correlation = df[df['epss_score'].notna()][['cvss', 'epss_score']].corr().iloc[0, 1]
print(f"\n CVSS-EPSS Correlation: {correlation:.3f}")
print(f"   (Weak correlation suggests they measure different aspects of risk)")


 CVSS-EPSS Correlation: 0.183
   (Weak correlation suggests they measure different aspects of risk)


## 8. Risk Signal Coverage

In [55]:
# Multi-signal analysis
signal_data = {
    'Signal': ['KEV', 'Healthcare', 'ATT&CK', 'CHPL', 'Curated'],
    'Count': [
        df['kev_flag'].sum(),
        df['is_healthcare'].sum(),
        df['attack_flag'].sum(),
        df['chpl_flag'].sum(),
        df['is_curated'].sum()
    ]
}
signal_df = pd.DataFrame(signal_data)
signal_df['Percentage'] = (signal_df['Count'] / len(df)) * 100

fig = px.bar(
    signal_df,
    x='Signal',
    y='Count',
    title='Risk Signal Coverage',
    text='Count',
    color='Signal',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(height=450, showlegend=False)

save_plot(fig, 'risk_signal_coverage')

print("\n[TARGET] Risk Signal Summary:")
display_sample(signal_df, n=10)


[TARGET] Risk Signal Summary:
Showing 10 of 5 rows


,Signal,Count,Percentage
0,KEV,1177,0.560084
1,Healthcare,796,0.378782
2,ATT&CK,81534,38.798555
3,CHPL,4951,2.355970
4,Curated,52,0.024745


In [56]:
# Multi-signal overlap analysis
print("\n High-Risk CVEs (Multiple Signals):")
print(f"  KEV + Healthcare: {(df['kev_flag'] & df['is_healthcare']).sum():,}")
print(f"  KEV + ATT&CK: {(df['kev_flag'] & df['attack_flag']).sum():,}")
print(f"  KEV + CHPL: {(df['kev_flag'] & df['chpl_flag']).sum():,}")
print(f"  Healthcare + CHPL: {(df['is_healthcare'] & df['chpl_flag']).sum():,}")
print(f"\n  Triple signal (KEV + Healthcare + ATT&CK): {(df['kev_flag'] & df['is_healthcare'] & df['attack_flag']).sum():,}")


 High-Risk CVEs (Multiple Signals):
  KEV + Healthcare: 2
  KEV + ATT&CK: 387
  KEV + CHPL: 43
  Healthcare + CHPL: 71

  Triple signal (KEV + Healthcare + ATT&CK): 0


## 9. Feature Correlations

In [57]:
# Correlation matrix for numeric features
numeric_cols = ['cvss', 'epss_score', 'epss_percentile', 'kev_flag', 
                'is_healthcare', 'attack_flag', 'chpl_flag']
corr_df = df[numeric_cols].corr()

fig = px.imshow(
    corr_df,
    text_auto='.2f',
    title='Feature Correlation Matrix',
    color_continuous_scale='RdBu_r',
    aspect='auto'
)
fig.update_layout(height=500)

save_plot(fig, 'feature_correlations')

print("\n[STATS] Key Correlations:")
# Find strongest correlations (excluding diagonal)
corr_pairs = []
for i in range(len(corr_df.columns)):
    for j in range(i+1, len(corr_df.columns)):
        corr_pairs.append((
            corr_df.columns[i],
            corr_df.columns[j],
            corr_df.iloc[i, j]
        ))

corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
for feat1, feat2, corr_val in corr_pairs[:5]:
    print(f"  {feat1:20s} <-> {feat2:20s}: {corr_val:+.3f}")


[STATS] Key Correlations:
  epss_score           <-> epss_percentile     : +0.416
  cvss                 <-> epss_percentile     : +0.377
  epss_score           <-> kev_flag            : +0.339
  cvss                 <-> epss_score          : +0.183
  epss_percentile      <-> kev_flag            : +0.130


## 10. CWE Analysis

In [58]:
# Extract primary CWE
df['cwe_primary'] = df['cwe'].str.extract(r'(CWE-\d+)')[0]

# Top CWEs
top_cwes = df['cwe_primary'].value_counts().head(15)

fig = px.bar(
    x=top_cwes.index,
    y=top_cwes.values,
    title='Top 15 CWEs (Common Weakness Enumeration)',
    labels={'x': 'CWE', 'y': 'Number of CVEs'},
    text=top_cwes.values
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(height=450, showlegend=False, xaxis_tickangle=-45)

save_plot(fig, 'top_cwes')

print(f"\n[STATS] CWE Statistics:")
print(f"  CVEs with CWE: {df['cwe_primary'].notna().sum():,} ({df['cwe_primary'].notna().mean()*100:.1f}%)")
print(f"  Unique CWEs: {df['cwe_primary'].nunique():,}")
print(f"\n  Top 10 CWEs:")
for cwe, count in top_cwes.head(10).items():
    print(f"    {cwe}: {count:,}")


[STATS] CWE Statistics:
  CVEs with CWE: 186,288 (88.6%)
  Unique CWEs: 717

  Top 10 CWEs:
    CWE-79: 31,647
    CWE-89: 9,948
    CWE-787: 8,960
    CWE-352: 6,763
    CWE-125: 6,506
    CWE-20: 6,089
    CWE-862: 5,672
    CWE-416: 5,312
    CWE-22: 5,091
    CWE-200: 4,400


## 11. Label Distribution Analysis

In [59]:
# Analyze weak label distribution
if 'label' in df.columns and df['label'].notna().sum() > 0:
    df_labeled = df[df['label'].notna()].copy()
    
    label_counts = df_labeled['label'].value_counts().sort_index()
    
    fig = px.bar(
        x=label_counts.index.astype(str),
        y=label_counts.values,
        title='Weak Label Distribution',
        labels={'x': 'Label (0=Low, 5=Critical)', 'y': 'Number of CVEs'},
        text=label_counts.values,
        color=label_counts.values,
        color_continuous_scale='YlOrRd'
    )
    fig.update_traces(texttemplate='%{text:,}', textposition='outside')
    fig.update_layout(height=400, showlegend=False)
    
    save_plot(fig, 'label_distribution')
    
    print("\n  Label Distribution:")
    for label, count in label_counts.items():
        pct = (count / len(df_labeled)) * 100
        print(f"  Label {label}: {count:6,} ({pct:5.2f}%)")
    
    # Label by signals
    print("\n  Average signals by label:")
    for label in sorted(df_labeled['label'].unique()):
        label_df = df_labeled[df_labeled['label'] == label]
        avg_kev = label_df['kev_flag'].mean()
        avg_hc = label_df['is_healthcare'].mean()
        avg_attack = label_df['attack_flag'].mean()
        print(f"    Label {label}: KEV={avg_kev:.2%}, HC={avg_hc:.2%}, ATT&CK={avg_attack:.2%}")
else:
    print("\n[WARN]  No labels found. Run Feature_Engineering notebook to generate weak labels.")


  Label Distribution:
  Label 0: 10,577 ( 5.03%)
  Label 1: 34,547 (16.44%)
  Label 2: 152,122 (72.39%)
  Label 3: 12,635 ( 6.01%)
  Label 4:    266 ( 0.13%)

  Average signals by label:
    Label 0: KEV=0.00%, HC=0.09%, ATT&CK=36.73%
    Label 1: KEV=0.00%, HC=0.08%, ATT&CK=42.18%
    Label 2: KEV=0.00%, HC=0.44%, ATT&CK=37.60%
    Label 3: KEV=7.21%, HC=0.70%, ATT&CK=44.65%
    Label 4: KEV=100.00%, HC=0.00%, ATT&CK=88.72%


## 12. Summary Statistics

In [60]:
print("="*70)
print("EDA SUMMARY")
print("="*70)
print(f"\n[STATS] Dataset:")
print(f"  Total CVEs: {len(df):,}")
print(f"  Date range: {df['published'].min().date()} to {df['published'].max().date()}")
print(f"  Avg CVEs/month: {len(df) / len(df_monthly):.1f}")

print(f"\n[WARN]  Severity:")
print(f"  Mean CVSS: {df['cvss'].mean():.2f}")
print(f"  Critical (9.0+): {(df['cvss'] >= 9.0).sum():,} ({(df['cvss'] >= 9.0).mean()*100:.1f}%)")

print(f"\n[TARGET] Risk Signals:")
print(f"  KEV: {df['kev_flag'].sum():,}")
print(f"  Healthcare: {df['is_healthcare'].sum():,}")
print(f"  ATT&CK: {df['attack_flag'].sum():,}")
print(f"  CHPL: {df['chpl_flag'].sum():,}")

print(f"\n All plots saved to: outputs/plots/")
print("="*70)

# Close database
db.conn.close()
print("\n[OK] EDA Complete")

EDA SUMMARY

[STATS] Dataset:
  Total CVEs: 210,147
  Date range: 2018-01-01 to 2025-12-31
  Avg CVEs/month: 2189.0

[WARN]  Severity:
  Mean CVSS: 6.88
  Critical (9.0+): 25,231 (12.0%)

[TARGET] Risk Signals:
  KEV: 1,177
  Healthcare: 796
  ATT&CK: 81,534
  CHPL: 4,951

 All plots saved to: outputs/plots/

[OK] EDA Complete


## 13. Enhanced Features Analysis

**Purpose**: Analyze the 37 enhanced features from `features_enhanced_latest.csv`

Enhanced feature categories:
- **CVSS Decomposition** (10): Individual CVSS v3.x dimensions (AV, AC, PR, UI, S, C, I, A)
- **CWE Intelligence** (8): CWE Top 25 detection, categories, severity scoring
- **Description NLP** (10): Exploitation keywords (RCE, auth bypass, SQLi, XSS, etc.)
- **Vendor Features** (3): High-risk vendor detection, healthcare vendors
- **Interaction Features** (6): Compound risk indicators (ultimate risk, critical exploitable)

In [61]:
# Load enhanced features dataset
features_path = project_root / "outputs" / "features" / "features_enhanced_latest.csv"

if features_path.exists():
    print("="*70)
    print("ENHANCED FEATURES DATASET")
    print("="*70)
    
    df_enhanced = pd.read_csv(features_path)
    
    print(f"\n[OK] Loaded enhanced features from: {features_path}")
    print(f"  Total CVEs: {len(df_enhanced):,}")
    print(f"  Total features: {len(df_enhanced.columns)}")
    print(f"  File size: {features_path.stat().st_size / 1024**2:.1f} MB")
    
    # Categorize features
    cvss_features = [c for c in df_enhanced.columns if c.startswith('cvss_')]
    cwe_features = [c for c in df_enhanced.columns if c.startswith('cwe_')]
    desc_features = [c for c in df_enhanced.columns if c.startswith('desc_')]
    vendor_features = [c for c in df_enhanced.columns if c.startswith('vendor_')]
    interaction_features = [c for c in df_enhanced.columns if any(x in c for x in ['ultimate', 'critical', 'network', 'auth'])]
    
    print(f"\n[TARGET] Enhanced Feature Categories:")
    print(f"  CVSS decomposition: {len(cvss_features)} features")
    print(f"  CWE intelligence: {len(cwe_features)} features")
    print(f"  Description NLP: {len(desc_features)} features")
    print(f"  Vendor features: {len(vendor_features)} features")
    print(f"  Interaction features: {len(interaction_features)} features")
    
    print("\n" + "="*70)
else:
    print(f"\n[WARN] Enhanced features file not found: {features_path}")
    print(f"[INFO] Run: python apply_enhanced_features.py after STEP_3")
    df_enhanced = None

ENHANCED FEATURES DATASET

[OK] Loaded enhanced features from: /Users/vinayksharma/AirDnd/cti_recommender/outputs/features/features_enhanced_latest.csv
  Total CVEs: 210,147
  Total features: 56
  File size: 69.8 MB

[TARGET] Enhanced Feature Categories:
  CVSS decomposition: 14 features
  CWE intelligence: 8 features
  Description NLP: 0 features
  Vendor features: 3 features
  Interaction features: 4 features



### 13.1 CVSS Decomposition Analysis

Individual CVSS v3.x dimensions provide granular attack vector insights.

In [62]:
if df_enhanced is not None:
    print("="*70)
    print("CVSS DECOMPOSITION ANALYSIS")
    print("="*70)
    
    cvss_dims = ['cvss_av', 'cvss_ac', 'cvss_pr', 'cvss_ui', 'cvss_s', 'cvss_c', 'cvss_i', 'cvss_a']
    
    if all(dim in df_enhanced.columns for dim in cvss_dims):
        print(f"\n[STATS] CVSS Dimension Coverage:")
        for dim in cvss_dims:
            coverage = (df_enhanced[dim].notna().sum() / len(df_enhanced)) * 100
            mean_val = df_enhanced[dim].mean()
            print(f"  {dim:15s}: {coverage:5.1f}% coverage, mean={mean_val:.2f}")
        
        # Attack Vector distribution
        if 'cvss_av' in df_enhanced.columns:
            av_dist = df_enhanced['cvss_av'].value_counts().sort_index(ascending=False)
            av_labels = {4: 'Network', 3: 'Adjacent', 2: 'Local', 1: 'Physical'}
            print(f"\n Attack Vector Distribution (cvss_av):")
            for val, count in av_dist.items():
                if pd.notna(val):
                    label = av_labels.get(int(val), f'Value {val}')
                    pct = (count / len(df_enhanced)) * 100
                    print(f"  {label:15s} (AV={int(val)}): {count:6,} ({pct:5.2f}%)")
        
        # Scope Changed
        if 'cvss_s' in df_enhanced.columns:
            scope_changed = df_enhanced['cvss_s'].sum()
            scope_pct = (scope_changed / len(df_enhanced)) * 100
            print(f"\n Scope Changed (cvss_s=1): {int(scope_changed):,} ({scope_pct:.2f}%)")
            print(f"  [INSIGHT] {scope_pct:.1f}% of CVEs allow scope escalation beyond vulnerable component")
        
        # CIA impact (High impact = 1)
        print(f"\n CIA Impact (High=1, Low/None=0):")
        impact_labels = {'cvss_c': 'Confidentiality', 'cvss_i': 'Integrity', 'cvss_a': 'Availability'}
        for impact, label in impact_labels.items():
            if impact in df_enhanced.columns:
                high_impact = df_enhanced[impact].sum()
                high_pct = (high_impact / len(df_enhanced)) * 100
                print(f"  {label:18s}: {int(high_impact):6,} ({high_pct:5.2f}%)")
    
    print("\n" + "="*70)

CVSS DECOMPOSITION ANALYSIS

[STATS] CVSS Dimension Coverage:
  cvss_av        : 100.0% coverage, mean=3.47
  cvss_ac        : 100.0% coverage, mean=1.91
  cvss_pr        : 100.0% coverage, mean=2.45
  cvss_ui        : 100.0% coverage, mean=1.67
  cvss_s         : 100.0% coverage, mean=1.21
  cvss_c         : 100.0% coverage, mean=2.53
  cvss_i         : 100.0% coverage, mean=2.12
  cvss_a         : 100.0% coverage, mean=2.10

 Attack Vector Distribution (cvss_av):
  Network         (AV=4): 151,912 (72.29%)
  Adjacent        (AV=3):  6,191 ( 2.95%)
  Local           (AV=2): 49,921 (23.76%)
  Physical        (AV=1):  2,123 ( 1.01%)

 Scope Changed (cvss_s=1): 254,722 (121.21%)
  [INSIGHT] 121.2% of CVEs allow scope escalation beyond vulnerable component

 CIA Impact (High=1, Low/None=0):
  Confidentiality   : 530,905 (252.64%)
  Integrity         : 444,573 (211.55%)
  Availability      : 442,161 (210.41%)



### 13.2 CWE Intelligence Analysis

CWE Top 25 and category-based weakness detection.

In [63]:
if df_enhanced is not None:
    print("="*70)
    print("CWE INTELLIGENCE ANALYSIS")
    print("="*70)
    
    # CWE Top 25 detection
    if 'cwe_is_top25' in df_enhanced.columns:
        top25_count = df_enhanced['cwe_is_top25'].sum()
        top25_pct = (top25_count / len(df_enhanced)) * 100
        print(f"\n CWE Top 25 Coverage:")
        print(f"  CVEs with Top 25 CWEs: {int(top25_count):,} ({top25_pct:.2f}%)")
        print(f"  [INSIGHT] Over half of all CVEs contain CWE Top 25 weaknesses")
    
    # CWE category distribution
    cwe_categories = {
        'cwe_is_injection': 'Injection', 
        'cwe_is_crypto': 'Cryptographic', 
        'cwe_is_access_control': 'Access Control',
        'cwe_is_input_validation': 'Input Validation', 
        'cwe_is_memory_corruption': 'Memory Corruption'
    }
    
    print(f"\n[TARGET] CWE Category Distribution:")
    cat_counts = []
    for cat, label in cwe_categories.items():
        if cat in df_enhanced.columns:
            count = df_enhanced[cat].sum()
            pct = (count / len(df_enhanced)) * 100
            cat_counts.append((label, int(count), pct))
            print(f"  {label:25s}: {int(count):6,} ({pct:5.2f}%)")
    
    # Top category
    if cat_counts:
        cat_counts.sort(key=lambda x: x[1], reverse=True)
        print(f"\n  [INSIGHT] Most common weakness: {cat_counts[0][0]} ({cat_counts[0][2]:.1f}% of CVEs)")
    
    # CWE severity
    if 'cwe_severity_score' in df_enhanced.columns:
        severity_mean = df_enhanced['cwe_severity_score'].mean()
        high_severity = (df_enhanced['cwe_severity_score'] >= 8).sum()
        high_sev_pct = (high_severity / len(df_enhanced)) * 100
        print(f"\n CWE Severity Score:")
        print(f"  Mean severity: {severity_mean:.2f}")
        print(f"  High severity (≥8): {int(high_severity):,} ({high_sev_pct:.2f}%)")
    
    print("\n" + "="*70)

CWE INTELLIGENCE ANALYSIS

 CWE Top 25 Coverage:
  CVEs with Top 25 CWEs: 121,165 (57.66%)
  [INSIGHT] Over half of all CVEs contain CWE Top 25 weaknesses

[TARGET] CWE Category Distribution:
  Injection                : 51,695 (24.60%)

  [INSIGHT] Most common weakness: Injection (24.6% of CVEs)

 CWE Severity Score:
  Mean severity: 2.04
  High severity (≥8): 0 (0.00%)



### 13.3 Description NLP Features Analysis

Exploitation keyword detection from CVE descriptions.

In [64]:
if df_enhanced is not None:
    print("="*70)
    print("DESCRIPTION NLP ANALYSIS")
    print("="*70)
    
    nlp_features = [
        ('desc_has_rce', 'Remote Code Execution'),
        ('desc_has_auth_bypass', 'Authentication Bypass'),
        ('desc_has_priv_esc', 'Privilege Escalation'),
        ('desc_has_sqli', 'SQL Injection'),
        ('desc_has_xss', 'Cross-Site Scripting'),
        ('desc_has_dos', 'Denial of Service'),
        ('desc_has_buffer_overflow', 'Buffer Overflow'),
        ('desc_has_path_traversal', 'Path Traversal'),
        ('desc_has_csrf', 'CSRF'),
        ('desc_has_xxe', 'XML External Entity')
    ]
    
    print(f"\n Exploitation Pattern Detection:")
    pattern_counts = []
    for feature, label in nlp_features:
        if feature in df_enhanced.columns:
            count = df_enhanced[feature].sum()
            pct = (count / len(df_enhanced)) * 100
            pattern_counts.append((label, int(count), pct))
            print(f"  {label:30s}: {int(count):6,} ({pct:5.2f}%)")
    
    # Top 3 exploitation patterns
    if pattern_counts:
        pattern_counts.sort(key=lambda x: x[1], reverse=True)
        print(f"\n[INSIGHT] Top 3 Exploitation Patterns:")
        for i, (label, count, pct) in enumerate(pattern_counts[:3], 1):
            print(f"  {i}. {label}: {count:,} CVEs ({pct:.2f}%)")
    
    print("\n" + "="*70)

DESCRIPTION NLP ANALYSIS

 Exploitation Pattern Detection:



### 13.4 Vendor & Interaction Features

High-risk vendor detection and compound risk indicators.

In [65]:
if df_enhanced is not None:
    print("="*70)
    print("VENDOR & INTERACTION FEATURES ANALYSIS")
    print("="*70)
    
    # Vendor features
    print(f"\n Vendor Features:")
    vendor_feats = [
        ('vendor_is_high_risk', 'High-Risk Vendor'),
        ('vendor_is_healthcare', 'Healthcare Vendor')
    ]
    
    for feature, label in vendor_feats:
        if feature in df_enhanced.columns:
            count = df_enhanced[feature].sum()
            pct = (count / len(df_enhanced)) * 100
            print(f"  {label:25s}: {int(count):6,} ({pct:5.2f}%)")
    
    if 'vendor_risk_score' in df_enhanced.columns:
        mean_score = df_enhanced['vendor_risk_score'].mean()
        high_risk = (df_enhanced['vendor_risk_score'] >= 2).sum()
        high_risk_pct = (high_risk / len(df_enhanced)) * 100
        print(f"  {'Vendor Risk Score':25s}: mean={mean_score:.2f}, high risk (≥2): {int(high_risk):,} ({high_risk_pct:.2f}%)")
    
    # Interaction features (compound risk)
    print(f"\n[WARN] Compound Risk Indicators:")
    interaction_feats = [
        ('ultimate_risk', 'Ultimate Risk (KEV + Network + No Auth)'),
        ('critical_exploitable', 'Critical Exploitable (CVSS≥9 + Network)'),
        ('network_accessible', 'Network Accessible'),
        ('auth_not_required', 'No Authentication Required')
    ]
    
    for feature, label in interaction_feats:
        if feature in df_enhanced.columns:
            count = df_enhanced[feature].sum()
            pct = (count / len(df_enhanced)) * 100
            print(f"  {label:50s}: {int(count):6,} ({pct:5.2f}%)")
    
    # Ultimate risk insight
    if 'ultimate_risk' in df_enhanced.columns:
        ultimate_count = df_enhanced['ultimate_risk'].sum()
        ultimate_pct = (ultimate_count / len(df_enhanced)) * 100
        print(f"\n[CRITICAL] Ultimate Risk CVEs: {int(ultimate_count):,} ({ultimate_pct:.2f}%)")
        print(f"  Actively exploited (KEV), network-accessible, no authentication required")
        print(f"  These require IMMEDIATE patching priority")
    
    print("\n" + "="*70)

VENDOR & INTERACTION FEATURES ANALYSIS

 Vendor Features:
  High-Risk Vendor         :      0 ( 0.00%)
  Healthcare Vendor        :      0 ( 0.00%)
  Vendor Risk Score        : mean=0.00, high risk (≥2): 0 (0.00%)

[WARN] Compound Risk Indicators:



### 13.5 Enhanced Features Summary

Key insights from 37 enhanced features.

In [66]:
if df_enhanced is not None:
    print("="*70)
    print("ENHANCED FEATURES SUMMARY")
    print("="*70)
    
    print(f"\n[STATS] Feature Coverage:")
    print(f"  Original basic features: 16")
    print(f"  Enhanced features added: 37")
    print(f"  Total features available: {len(df_enhanced.columns)}")
    print(f"  CVEs analyzed: {len(df_enhanced):,}")
    
    # Key insights
    print(f"\n[INSIGHT] Key Findings:")
    
    if 'cvss_av' in df_enhanced.columns:
        network_accessible = (df_enhanced['cvss_av'] == 4).sum()
        net_pct = (network_accessible / len(df_enhanced)) * 100
        print(f"  1. {net_pct:.1f}% of CVEs are network-accessible (Attack Vector: Network)")
    
    if 'cwe_is_top25' in df_enhanced.columns:
        top25_pct = (df_enhanced['cwe_is_top25'].sum() / len(df_enhanced)) * 100
        print(f"  2. {top25_pct:.1f}% contain CWE Top 25 weaknesses")
    
    if 'ultimate_risk' in df_enhanced.columns:
        ultimate_pct = (df_enhanced['ultimate_risk'].sum() / len(df_enhanced)) * 100
        print(f"  3. {ultimate_pct:.2f}% are 'ultimate risk' (KEV + Network + No Auth)")
    
    if 'desc_has_rce' in df_enhanced.columns:
        rce_pct = (df_enhanced['desc_has_rce'].sum() / len(df_enhanced)) * 100
        print(f"  4. {rce_pct:.1f}% mention Remote Code Execution in description")
    
    if 'cvss_s' in df_enhanced.columns:
        scope_pct = (df_enhanced['cvss_s'].sum() / len(df_enhanced)) * 100
        print(f"  5. {scope_pct:.1f}% allow scope escalation beyond vulnerable component")
    
    print(f"\n[OK] Enhanced features ready for STEP_4 (Model Training)")
    print("="*70)

ENHANCED FEATURES SUMMARY

[STATS] Feature Coverage:
  Original basic features: 16
  Enhanced features added: 37
  Total features available: 56
  CVEs analyzed: 210,147

[INSIGHT] Key Findings:
  1. 72.3% of CVEs are network-accessible (Attack Vector: Network)
  2. 57.7% contain CWE Top 25 weaknesses
  5. 121.2% allow scope escalation beyond vulnerable component

[OK] Enhanced features ready for STEP_4 (Model Training)


## Next Steps

1. **Feature Engineering** -> Run `Feature_Engineering.ipynb` to create ML features
2. **Model Training** -> Run `Model_Training_And_Evaluation.ipynb` to train and compare models

---